# 2d. Data schemas: `conformsTo` generated here or pointing elsewhere

`dct:conformsTo` is not always a schema. DCAT 3 (§6.4.2) defines it on any resource as *"an
established standard to which the described resource conforms"*: a model, schema, ontology, view
or profile. DCAT-AP 3.0 names it per class: on a **Dataset** it is *"an implementing rule or other
specification"*, on a **Distribution** it is *"linked schemas"*: *"an established schema to which
the described Distribution conforms"*.

The catalog keeps a distinguished **data schema** next to the other standards:

- A dataset or distribution points to it with the `schema` term of the management API. It is
  either a **JSON Schema hosted by this connector** (generated here, each version immutable at its
  own public URL) or an **external URL**.
- The profiles (DSP, DCAT 3, DCAT-AP) publish it as `dct:conformsTo`, typed `dct:Standard`. A
  distribution with no schema of its own **inherits its dataset's**; one with its own overrides it.
- Writing a resource whose `schema` names a hosted version that does not exist is refused, and a
  version some resource points to cannot be deleted.

This notebook infers a schema from sample readings, hosts it, points a dataset to it, reads it back
as consumers see it, publishes a second version and cleans up. Provider only; no onboarding needed.

## 1. Configuration and helpers

`show(response)` prints an HTTP answer and fails the cell on an unexpected status, so a broken step
stops the notebook. `conforms_to(document)` lists the `conformsTo` of every node of a projection.

In [ ]:
import json
import os

import requests

data_space_provider = os.environ.get("PROVIDER_URL", "http://127.0.0.1:1200")
api = f"{data_space_provider}/api/v1/catalog-agent"
CATALOG_CTX = "https://w3id.org/eunomia/v1/catalog-context.jsonld"


def show(response, ok=True, limit=1500):
    """Prints an HTTP answer (cut to `limit` characters); with ok, raises unless it is 2xx."""
    try:
        text = json.dumps(response.json(), indent=2, ensure_ascii=False)
    except ValueError:
        text = response.text
    print(f"HTTP {response.status_code} {response.request.method} {response.url}")
    print(text if len(text) <= limit else text[:limit] + f"\n… ({len(text)} characters)")
    if ok:
        response.raise_for_status()
    return response


def conforms_to(document):
    """`{node id: [conformsTo IRIs]}` over the nodes of a projected document."""
    nodes = document.get("@graph", [document])
    found = {}
    for node in nodes:
        for key in ("conformsTo", "dct:conformsTo"):
            values = node.get(key)
            if values is None:
                continue
            values = values if isinstance(values, list) else [values]
            found[node["@id"]] = [v["@id"] if isinstance(v, dict) else v for v in values]
    return found

In [ ]:
main_catalog_id = show(requests.get(f"{api}/catalogs/main"), limit=300).json()["@id"]
main_service_id = show(requests.get(f"{api}/data-services/main"), limit=300).json()["@id"]

## 2. Infer a schema from samples (`POST /schemas/infer`)

Each sample is a whole document. Fields present in every sample come out **required**; integer
and decimal values together make a `number`; strings that clearly are a date-time, date, email,
URI or UUID keep that `format`. Nothing is stored.

In [ ]:
samples = [
    {"station": "MAD-01", "at": "2026-10-29T10:00:00Z", "temperature": 14.5, "humidity": 61,
     "location": {"lat": 40.41, "lon": -3.70}, "note": "clear"},
    {"station": "BCN-02", "at": "2026-10-29T10:00:00Z", "temperature": 18, "humidity": 70,
     "location": {"lat": 41.39, "lon": 2.17}},
]
inferred = show(requests.post(f"{api}/schemas/infer", json={"samples": samples}), limit=2500).json()
print("required:", inferred["required"])

## 3. Check it, and samples against it (`POST /schemas/validate`)

The schema is checked against the JSON Schema 2020-12 meta-schema, then each sample against the
schema. A reading with a text temperature fails at `/temperature`.

In [ ]:
inferred["properties"]["temperature"]["description"] = "Degrees Celsius"
bad = dict(samples[1], temperature="warm")
report = show(requests.post(f"{api}/schemas/validate", json={"schema": inferred, "samples": [samples[0], bad]})).json()
for sample in report["samples"]:
    print(sample["index"], "valid" if sample["valid"] else [i["path"] for i in sample["issues"]])

## 4. Host it (`POST /schemas`)

A new schema is **version 1** of a fresh `urn:schema:…`. Its public URL is what `conformsTo` will
point to; the stored content drops any `$id` it came with, and the served one names that URL.
`visibility: Public` lets anyone dereference it; a private schema is not served.

In [ ]:
created = show(requests.post(f"{api}/schemas", json={
    "title": "Weather readings",
    "description": "One reading of a weather station",
    "visibility": "Public",
    "schema": inferred,
})).json()
schema_id, schema_v1 = created["id"], created["url"]
print(schema_v1)

## 5. Dereference it with no token (`GET /public/schemas/{id}/{version}`)

`application/schema+json`, `Cache-Control: immutable` (a version never changes) and CORS open, so
a consumer, a validator or a browser can fetch it.

In [ ]:
response = requests.get(schema_v1)
for header in ["Content-Type", "Cache-Control", "Access-Control-Allow-Origin"]:
    print(f"{header}: {response.headers.get(header)}")
print(json.dumps({k: response.json()[k] for k in ["$schema", "$id", "title", "required"]}, indent=2))

## 6. A dataset that conforms to it, and two distributions

The dataset sets `schema`. The pull distribution sets none, so it **inherits** the dataset's; the
push one sends GeoJSON features and **overrides** it with an external schema.

In [ ]:
dataset_id = show(requests.post(f"{api}/datasets", json={
    "@context": CATALOG_CTX,
    "@type": "Dataset",
    "title": {"en": "Weather readings (schema tutorial)"},
    "schema": schema_v1,
    "inCatalog": main_catalog_id,
}), limit=600).json()["@id"]

distribution = lambda title, fmt, extra: {
    "@context": CATALOG_CTX, "@type": "Distribution", "title": {"en": title}, "format": fmt,
    "accessService": [main_service_id], "ofDataset": dataset_id, **extra,
}
pull_id = requests.post(f"{api}/distributions", json=distribution("Readings, pull", "http+pull", {})).json()["@id"]
geojson = "https://geojson.org/schema/FeatureCollection.json"
push_id = requests.post(f"{api}/distributions", json=distribution("Readings, GeoJSON push", "http+push", {"schema": geojson})).json()["@id"]
print(dataset_id, pull_id, push_id, sep="\n")

## 7. As consumers see it: `conformsTo`, inherited

The public DCAT-AP and DCAT 3 routes project the dataset with its distributions. The internal
`schema` term never shows: it comes out as `dct:conformsTo`, and the pull distribution carries the
dataset's schema because it has none of its own.

In [ ]:
for profile in ["dcat-ap", "dcat3"]:
    projected = requests.get(f"{api}/public/{profile}/resources/{dataset_id}")
    projected.raise_for_status()
    print(profile)
    for node, standards in conforms_to(projected.json()).items():
        label = {dataset_id: "dataset", pull_id: "pull (inherited)", push_id: "push (own)"}.get(node, node)
        print(f"  {label:18} {standards}")

The DSP projection a peer receives carries them too; the DSP and DCAT-AP doctors still find the
dataset compliant:

In [ ]:
doctor = requests.get(f"{api}/datasets/{dataset_id}/doctor", params={"deep": "true"}).json()
for diagnosis in doctor["profiles"]:
    print(f"{diagnosis['profile']:8} compliant={diagnosis['compliant']} violations={diagnosis['violations']}")

## 8. Versions are immutable (`POST /schemas/{id}/versions`)

A change is a **new version** at a new URL; version 1 keeps its content, so what a consumer agreed
to never changes underneath. While the dataset points to v1, deleting v1 is a `409`.

In [ ]:
inferred["properties"]["station"]["description"] = "Station code, e.g. MAD-01"
v2 = show(requests.post(f"{api}/schemas/{schema_id}/versions", json={"schema": inferred})).json()
schema_v2 = v2["url"]

show(requests.delete(f"{api}/schemas/{schema_id}/versions/1"), ok=False)
usage = requests.get(f"{api}/schemas/{schema_id}/versions/1").json()["usedBy"]
print("v1 is used by:", [(u["kind"], u["title"]) for u in usage])

Moving the dataset to v2 is an ordinary replace: read it with its `ETag`, set `schema`, send it
back with `If-Match`. Then v1 can go.

In [ ]:
current = requests.get(f"{api}/datasets/{dataset_id}")
document = {k: v for k, v in current.json().items() if k not in ("issued", "modified")}
document["schema"] = schema_v2
show(requests.put(f"{api}/datasets/{dataset_id}", json=document, headers={"If-Match": current.headers["ETag"]}), limit=400)
show(requests.delete(f"{api}/schemas/{schema_id}/versions/1"))
print([v["version"] for v in requests.get(f"{api}/schemas/{schema_id}").json()])

## 9. External schemas and wrong references

`POST /schemas/check-url` fetches a URL and says whether it holds a JSON Schema; it only reports,
nothing is stored. A `schema` naming a **hosted** version that does not exist is refused when the
resource is written; external URLs are taken as given.

In [ ]:
print(json.dumps(requests.post(f"{api}/schemas/check-url", json={"url": geojson}).json(), indent=2)[:600])

current = requests.get(f"{api}/datasets/{dataset_id}")
document = {k: v for k, v in current.json().items() if k not in ("issued", "modified")}
document["schema"] = schema_v2.rsplit("/", 1)[0] + "/99"
show(requests.put(f"{api}/datasets/{dataset_id}", json=document, headers={"If-Match": current.headers["ETag"]}), ok=False)

## 10. Clean up

Deleting the dataset takes its distributions with it; then nothing points to the schema and it
can be deleted whole.

In [ ]:
show(requests.delete(f"{api}/datasets/{dataset_id}"))
show(requests.delete(f"{api}/schemas/{schema_id}"))